In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath("../src"))
from data_processing import load_and_prepare_data, split_train_validation, create_mappings
from evaluation import evaluate_recommender

df = load_and_prepare_data("../data/processed/online_retail_clean.parquet")

grouped_descs = df.groupby('StockCode')['Description']

def merge_unique_words(series_de_descricoes):
    # 1. Pega todas as descrições únicas (ex: {'a b', 'a c'})
    unique_descs_set = set(series_de_descricoes.unique())
    
    # 2. Junta tudo em uma única string (ex: "a b a c")
    full_string = ' '.join(unique_descs_set)
    
    # 3. Divide em palavras, pega o set de palavras únicas (ex: {'a', 'b', 'c'})
    unique_words_set = set(full_string.split())
    
    # 4. Re-junta na "super-descrição" final e limpa
    return ' '.join(unique_words_set)

catalogo_series = grouped_descs.apply(merge_unique_words)
catalogo_df = catalogo_series.reset_index()
catalogo_df.columns = ['StockCode', 'Description']
catalogo_df.to_parquet("../data/processed/catalogo_unico.parquet", index=False)
print(f"Quantidade de descrições originais: {len(df.Description.unique())}")
print(f"Quantidade de descrições únicas por item depois de merge das descrições repetidas: {len(catalogo_df)}")

Quantidade de descrições originais: 3860
Quantidade de descrições únicas por item depois de merge das descrições repetidas: 3659


In [2]:
tf_idf = TfidfVectorizer(
    stop_words='english', 
    lowercase=True, 
    ngram_range=(1, 2),
    max_df=0.5,
    sublinear_tf=True
)
tf_idf_matrix = tf_idf.fit_transform(catalogo_df['Description'])

cosine_sim = cosine_similarity(tf_idf_matrix)
cosine_sim_df = pd.DataFrame(cosine_sim, index=catalogo_df.index, columns=catalogo_df.index)

print(cosine_sim_df.shape)

item_map_content = {stock_code: i for i, stock_code in enumerate(catalogo_df['StockCode'])}
inverse_item_map_content = {i: stock_code for stock_code, i in item_map_content.items()}

(3659, 3659)


In [3]:
def get_similar_items_by_stockcode(stockcode, cosine_sim_df, catalogo_df, top_n=10):
    # encontra o índice numérico do catalogo_df correspondente ao StockCode
    matches = catalogo_df.index[catalogo_df['StockCode'] == stockcode].tolist()
    if not matches:
        return pd.Series(dtype=float)  # série vazia
    idx = matches[0]

    # usa o índice numérico para consultar a matriz de similaridade
    sim_scores = cosine_sim_df.loc[idx].sort_values(ascending=False)
    sim_scores = sim_scores.drop(idx, errors='ignore')  # remove o próprio item se presente
    top_items = sim_scores.head(top_n)
    return top_items

item_id = '22778'
similar_items = get_similar_items_by_stockcode(item_id, cosine_sim_df, catalogo_df, top_n=10)

if similar_items.empty:
    print(f"No item with StockCode {item_id} found.")
else:
    df_similar_items = catalogo_df.loc[similar_items.index].copy()
    df_similar_items['similarity'] = similar_items.values
    # mostra a linha do catálogo correspondente ao StockCode consultado
    print(catalogo_df[catalogo_df['StockCode'] == item_id])
    print(df_similar_items)

     StockCode                  Description
1627     22778  BELL JAR CLOCHE GLASS SMALL
     StockCode                         Description  similarity
1626     22777         BELL JAR CLOCHE GLASS LARGE    0.808401
3261     85146  BELL JAR GLASS JARDIN ETCHED SMALL    0.353546
3260     85145  BELL JAR GLASS JARDIN LARGE ETCHED    0.307925
2700     72811       ZINC/GLASS SMALL CANDLEHOLDER    0.256122
217      20803        SUNDAE GLASS SMALL DISH PINK    0.229485
3236     85125   ROUND GLASS SMALL CUT CANDLESTICK    0.216036
3237     85127  GLASS SMALL SQUARE CUT CANDLESTICK    0.214346
1924     23089                       GLASS BON JAR    0.153183
1238     22363                 GLASS MARMALADE JAR    0.147260
1998     23167       JAR SMALL TOP CERAMIC STORAGE    0.139474


In [4]:
print("\n--- Parte 3: Construindo o Campo de Testes 5-Core ---")

# 1. Divida os dados PRIMEIRO (evita data leakage)
train_df, validation_df = split_train_validation(df)

# 2. Encontre as entidades "Core" (APENAS do treino)
min_interactions = 5
user_counts = train_df['CustomerID'].value_counts()
item_counts = train_df['StockCode'].value_counts()

core_users = user_counts[user_counts >= min_interactions].index
core_items = item_counts[item_counts >= min_interactions].index

# 3. Filtre AMBOS os dataframes para criar o novo "universo"
train_df_f = train_df[
    (train_df['CustomerID'].isin(core_users)) &
    (train_df['StockCode'].isin(core_items))
].copy()

validation_df_f = validation_df[
    (validation_df['CustomerID'].isin(core_users)) &
    (validation_df['StockCode'].isin(core_items))
].copy()

# 4. Crie os Dicionários de Avaliação (Gabarito e Exclusão)
train_user_items_f = (
    train_df_f.groupby('CustomerID')['StockCode'].apply(set).to_dict()
)
validation_user_items_f = (
    validation_df_f.groupby('CustomerID')['StockCode'].apply(set).to_dict()
)

# 5. Crie a Lista Final de Usuários para a "Prova"
valid_users_for_eval_f = [
    u for u in validation_user_items_f
    if u in train_user_items_f
]


--- Parte 3: Construindo o Campo de Testes 5-Core ---


In [5]:
from typing import List, Dict, Set

print("\n--- Parte 4: Definindo a Função de Recomendação ---")

def create_content_recommender_fn(
    train_user_items: Dict[int, Set[str]],
    content_similarity_matrix: np.ndarray,
    item_map: Dict[str, int],
    inverse_item_map: Dict[int, str]
):
    """
    Cria e retorna uma função de recomendação (closure) que já tem
    acesso às matrizes e mapas necessários.
    """
    
    def recommend_fn(
        user_id: int, 
        K: int, 
        exclude: Set[str] = None 
    ) -> List[str]:
        
        # 1. Pegue o histórico de compras do usuário (do treino)
        history_stockcodes = train_user_items.get(user_id)
        if not history_stockcodes:
            return []  # Usuário sem histórico no treino

        # 2. Converta os StockCodes do histórico para os índices da matriz
        history_indices = []
        for stock_code in history_stockcodes:
            # Apenas considere itens que existem no nosso mapa de conteúdo
            idx = item_map.get(stock_code) 
            if idx is not None:
                history_indices.append(idx)
        
        if not history_indices:
            return [] # Itens do histórico não estão no catálogo de conteúdo

        # 3. Some os vetores de similaridade
        aggregated_scores = content_similarity_matrix[history_indices].sum(axis=0)

        # 4. Fator de novidade: 1.0 = trata igual; 0.5 = rebaixa itens vistos; 0.0 = exclui
        penalty_factor = 1

        # Aplica penalidade suave aos itens já comprados
        aggregated_scores[history_indices] *= penalty_factor
        
        # 5. Zere os scores de outros itens a excluir (o avaliador cuida disso)
        if exclude: 
            exclude_indices = [
                item_map.get(sc) for sc in exclude 
                if item_map.get(sc) is not None
            ]
            aggregated_scores[exclude_indices] = 0

        # 6. Pegue os K maiores scores
        top_k_indices = np.argsort(aggregated_scores)[-K:]

        # 7. Converta os índices de volta para StockCodes e reverta a lista
        top_k_stockcodes = [inverse_item_map[idx] for idx in top_k_indices][::-1]

        return top_k_stockcodes

    # Retorna a função interna que o avaliador usará
    return recommend_fn

print("Função 'create_content_recommender_fn' definida.")


--- Parte 4: Definindo a Função de Recomendação ---
Função 'create_content_recommender_fn' definida.


In [6]:
train_user_items_f

{12347: {'16008',
  '17021',
  '20665',
  '20719',
  '20780',
  '20782',
  '20966',
  '21035',
  '21041',
  '21064',
  '21154',
  '21171',
  '21265',
  '21578',
  '21636',
  '21731',
  '21791',
  '21832',
  '21975',
  '21976',
  '22131',
  '22134',
  '22195',
  '22196',
  '22212',
  '22252',
  '22371',
  '22372',
  '22374',
  '22375',
  '22376',
  '22417',
  '22422',
  '22423',
  '22432',
  '22492',
  '22494',
  '22497',
  '22550',
  '22561',
  '22621',
  '22697',
  '22698',
  '22699',
  '22725',
  '22726',
  '22727',
  '22728',
  '22729',
  '22771',
  '22772',
  '22773',
  '22774',
  '22775',
  '22805',
  '22821',
  '22945',
  '22992',
  '23076',
  '23084',
  '23146',
  '23147',
  '23162',
  '23170',
  '23171',
  '23172',
  '23173',
  '23174',
  '23175',
  '23177',
  '23297',
  '23308',
  '23316',
  '23420',
  '23421',
  '23422',
  '23480',
  '23503',
  '23506',
  '23508',
  '47559B',
  '47567B',
  '47580',
  '51014C',
  '71477',
  '84558A',
  '84559A',
  '84559B',
  '84625A',
  '8462

In [7]:
print("\n--- Parte 5: Avaliando o Modelo de Conteúdo (TF-IDF) ---")

# 1. Função de recomendação 
rec_fn_content = create_content_recommender_fn(
    train_user_items_f,         # histórico de treino (filtrado)
    cosine_sim,  # Matriz TF-IDF
    item_map_content,           # mapa de StockCode -> índice
    inverse_item_map_content    # mapa de índice -> StockCode
)

# 2. Executando a avaliação
content_rows = []
empty_dict = {}
for k in [5, 10, 20]:
    print(f"Avaliando para K={k}...")
    resk = evaluate_recommender(
        valid_users_for_eval_f,      # Usuários do experimento filtrado
        rec_fn_content,              # Função de recomendação de conteúdo
        validation_user_items_f,     # Gabarito filtrado
        empty_dict,          # Exclusão filtrada vazia
        k=k
    )
    content_rows.append({
        'K': k, 
        'Precision': resk[f'P@{k}'], 
        'Recall': resk[f'R@{k}'], 
        'NDCG': resk[f'NDCG@{k}']
    })

# 3. Os resultados
df_content_results = pd.DataFrame(content_rows).set_index('K')

print("\n--- 📊 Resultados Finais - Modelo de Conteúdo (TF-IDF) Puro ---")
print(df_content_results)


--- Parte 5: Avaliando o Modelo de Conteúdo (TF-IDF) ---
Avaliando para K=5...
Avaliando para K=10...
Avaliando para K=20...

--- 📊 Resultados Finais - Modelo de Conteúdo (TF-IDF) Puro ---
    Precision    Recall      NDCG
K                                
5    0.151223  0.070972  0.162749
10   0.134396  0.116072  0.163142
20   0.115382  0.183868  0.175588


# Preparando para o blending

In [8]:
import joblib

# O nome do arquivo que vamos criar
model_filename = '../models/cosine_sim.joblib'

# Salva o seu objeto 'best_model' no arquivo
print(f"Salvando o cosine_sim em {model_filename}...")
joblib.dump(cosine_sim, model_filename)

print("Modelo salvo com sucesso!")

Salvando o cosine_sim em ../models/cosine_sim.joblib...
Modelo salvo com sucesso!


In [9]:
# 1. Pegue o histórico de compras do usuário (do treino)
print(validation_df_f["CustomerID"])
user_id = 16048

history_stockcodes = train_user_items_f.get(user_id, [])

# 2. Converta os StockCodes do histórico para os índices da matriz
history_indices = []
for stock_code in history_stockcodes:
    # Apenas considere itens que existem no nosso mapa de conteúdo
    idx = item_map_content.get(stock_code) 
    if idx is not None:
        history_indices.append(idx)

num_content_items = cosine_sim.shape[0] 

if not history_indices:
    # Se o usuário não tem histórico, o score de conteúdo é ZERO para tudo
    scores = np.zeros(num_content_items)
else:
    scores = cosine_sim[history_indices].max(axis=0)


1571      16048
3247      17850
3248      17850
5442      14813
5443      14813
          ...  
391143    12680
391144    12680
391145    12680
391146    12680
391147    12680
Name: CustomerID, Length: 57109, dtype: int64
